# Data Cleaning & Preprocessing — CS Students Dataset — `cs_students.csv`

This notebook cleans and preprocesses the raw CS students dataset,
preparing it for analysis or machine learning.

---

## Import Libraries

In [1]:
import pandas as pd
import re

print('Libraries imported successfully!')

Libraries imported successfully!


## Load the Data & Inspect It

In [2]:
df = pd.read_csv('cs_students.csv')

print(f'Shape: {df.shape}')    # (rows, columns)
print()
print('Columns:', df.columns.tolist())
print()
df.head(5)

Shape: (180, 12)

Columns: ['Student ID', 'Name', 'Gender', 'Age', 'GPA', 'Major', 'Interested Domain', 'Projects', 'Future Career', 'Python', 'SQL', 'Java']



,Student ID,Name,Gender,Age,GPA,Major,Interested Domain,Projects,Future Career,Python,SQL,Java
0,1,John Smith,Male,21,3.5,Computer Science,Artificial Intelligence,Chatbot Development,Machine Learning Researcher,Strong,Strong,Weak
1,2,Alice Johnson,Female,20,3.2,Computer Science,Data Science,Data Analytics,Data Scientist,Average,Strong,Weak
2,3,Robert Davis,Male,22,3.8,Computer Science,Software Development,E-commerce Website,Software Engineer,Strong,Strong,Average
3,4,Emily Wilson,Female,21,3.7,Computer Science,Web Development,Full-Stack Web App,Web Developer,Weak,Strong,Strong
4,5,Michael Brown,Male,23,3.4,Computer Science,Cybersecurity,Network Security,Information Security Analyst,Average,Weak,Strong


## Check Data Types

In [3]:
df.dtypes

Student ID             int64
Name                  object
Gender                object
Age                    int64
GPA                  float64
Major                 object
Interested Domain     object
Projects              object
Future Career         object
Python                object
SQL                   object
Java                  object
dtype: object

## Check for Missing Values

We check how many `NaN` values exist in each column.

In [4]:
print('Missing values per column:')
print(df.isnull().sum())
print()
print(f'Total missing values: {df.isnull().sum().sum()}')

Missing values per column:
Student ID           0
Name                 0
Gender               0
Age                  0
GPA                  0
Major                0
Interested Domain    0
Projects             0
Future Career        0
Python               0
SQL                  0
Java                 0
dtype: int64

Total missing values: 0


## Handle Missing Values

The dataset has no missing values. However, we still apply a defensive fill
to handle any edge cases — filling numeric columns with their median,
and text columns with `'Unknown'`.

In [5]:
# Fill numeric NaNs with column median
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill text NaNs with 'Unknown'
str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].fillna('Unknown')

print('Missing values after filling:')
print(df.isnull().sum())

Missing values after filling:
Student ID           0
Name                 0
Gender               0
Age                  0
GPA                  0
Major                0
Interested Domain    0
Projects             0
Future Career        0
Python               0
SQL                  0
Java                 0
dtype: int64


## Remove Duplicates

In [6]:
print(f'Duplicates found: {df.duplicated().sum()}')

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Shape after removing duplicates: {df.shape}')

Duplicates found: 0
Shape after removing duplicates: (180, 12)


## Clean String Columns

We strip any leading or trailing whitespace from all text columns.
This prevents issues where `'Alice'` and `'Alice '` are treated as different values.

In [7]:
str_cols = df.select_dtypes(include='object').columns
print('String columns:', str_cols.tolist())

for col in str_cols:
    df[col] = df[col].str.strip()

print(f'Stripped whitespace from: {str_cols.tolist()}')

String columns: ['Name', 'Gender', 'Major', 'Interested Domain', 'Projects', 'Future Career', 'Python', 'SQL', 'Java']
Stripped whitespace from: ['Name', 'Gender', 'Major', 'Interested Domain', 'Projects', 'Future Career', 'Python', 'SQL', 'Java']


## Normalize Text Columns

We normalize the `Name`, `Major`, `Interested Domain`, and `Future Career` columns
to title case so values like `'machine learning'` and `'Machine Learning'`
are treated as the same.

In [8]:
text_cols = ['Name', 'Major', 'Interested Domain', 'Future Career', 'Projects']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print('Sample after normalization:')
print(df[text_cols].head(5))

Sample after normalization:
            Name             Major        Interested Domain  \
0     John Smith  Computer Science  Artificial Intelligence   
1  Alice Johnson  Computer Science             Data Science   
2   Robert Davis  Computer Science     Software Development   
3   Emily Wilson  Computer Science          Web Development   
4  Michael Brown  Computer Science            Cybersecurity   

                  Future Career             Projects  
0   Machine Learning Researcher  Chatbot Development  
1                Data Scientist       Data Analytics  
2             Software Engineer   E-Commerce Website  
3                 Web Developer   Full-Stack Web App  
4  Information Security Analyst     Network Security  


## Encode Ordinal Skill Columns

The skill columns (`Python`, `SQL`, `Java`) use ordinal text values:
`Weak`, `Average`, `Strong`. These have a natural order, so we map
them to integers: `Weak → 1`, `Average → 2`, `Strong → 3`.

This makes the data usable for numeric operations and machine learning models.

In [9]:
skill_map  = {'Weak': 1, 'Average': 2, 'Strong': 3}
skill_cols = ['Python', 'SQL', 'Java']

print('Skill columns before encoding:')
print(df[skill_cols].head(5))

for col in skill_cols:
    df[col] = df[col].map(skill_map)

print('\nSkill columns after encoding:')
print(df[skill_cols].head(5))
print()
print('Data types:', df[skill_cols].dtypes.tolist())

Skill columns before encoding:
    Python     SQL     Java
0   Strong  Strong     Weak
1  Average  Strong     Weak
2   Strong  Strong  Average
3     Weak  Strong   Strong
4  Average    Weak   Strong

Skill columns after encoding:
   Python  SQL  Java
0       3    3     1
1       2    3     1
2       3    3     2
3       1    3     3
4       2    1     3

Data types: [dtype('int64'), dtype('int64'), dtype('int64')]


## Encode the `Gender` Column

`Gender` is a binary categorical variable. We encode it as:
- `Male   → 0`
- `Female → 1`

In [10]:
print('Gender before encoding:')
print(df['Gender'].value_counts())

df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

print('\nGender after encoding:')
print(df['Gender'].value_counts())

Gender before encoding:
Gender
Male      102
Female     78
Name: count, dtype: int64

Gender after encoding:
Gender
0    102
1     78
Name: count, dtype: int64


## Check for Inconsistencies

We check for logical inconsistencies in the data:
- `Age` should be within a realistic student range (18–30)
- `GPA` should be between 0.0 and 4.0
- `Student ID` should be unique

In [11]:
# Age check
age_issues = df[(df['Age'] < 18) | (df['Age'] > 30)]
print(f'Rows with unusual age (< 18 or > 30): {len(age_issues)}')

# GPA check
gpa_issues = df[(df['GPA'] < 0.0) | (df['GPA'] > 4.0)]
print(f'Rows with invalid GPA (< 0 or > 4): {len(gpa_issues)}')

# Duplicate Student ID check
dup_ids = df['Student ID'].duplicated().sum()
print(f'Duplicate Student IDs: {dup_ids}')

if len(age_issues) == 0 and len(gpa_issues) == 0 and dup_ids == 0:
    print('\nNo logical inconsistencies found — data looks clean!')

Rows with unusual age (< 18 or > 30): 7
Rows with invalid GPA (< 0 or > 4): 0
Duplicate Student IDs: 0


## Check for Outliers

We use the **IQR method** to detect outliers in the `GPA` and `Age` columns.
Any entry with a value far outside the expected range may indicate a data error.

The IQR rule flags values below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR` as outliers.

In [12]:
for col in ['GPA', 'Age']:
    Q1  = df[col].quantile(0.25)
    Q3  = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'Outliers in {col}: {len(outliers)}  (bounds: {lower:.2f} – {upper:.2f})')
    if len(outliers) > 0:
        print(outliers[['Student ID', 'Name', col]])

Outliers in GPA: 0  (bounds: 3.20 – 4.00)
Outliers in Age: 9  (bounds: 19.12 – 24.12)
    Student ID                 Name  Age
76          77        Agent Coulson   35
77          78           Agent Hill   33
78          79    Agent Melinda May   37
79          80  Agent Daisy Johnson   29
80          81        Agent Simmons   31
81          82           Agent Fitz   35
82          83           Agent Mack   37
83          84    Agent Bobbi Morse   29
84          85   Agent Lance Hunter   33


## Add a Derived Feature: `Skill Score`

We create a new column `Skill Score` that is the average of the three skill columns.
This gives a single number representing the student's overall technical strength.

In [13]:
df['Skill Score'] = df[['Python', 'SQL', 'Java']].mean(axis=1).round(2)

print('Skill Score distribution:')
print(df['Skill Score'].describe())
print()
print('Sample:')
print(df[['Name', 'Python', 'SQL', 'Java', 'Skill Score']].head(5))

Skill Score distribution:
count    180.000000
mean       2.090833
std        0.230803
min        1.670000
25%        2.000000
50%        2.000000
75%        2.000000
max        2.670000
Name: Skill Score, dtype: float64

Sample:
            Name  Python  SQL  Java  Skill Score
0     John Smith       3    3     1         2.33
1  Alice Johnson       2    3     1         2.00
2   Robert Davis       3    3     2         2.67
3   Emily Wilson       1    3     3         2.33
4  Michael Brown       2    1     3         2.00


## Final Check

In [14]:
print(f'Final shape: {df.shape}')
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())
print()
df.head(5)

Final shape: (180, 13)

Data types:
Student ID             int64
Name                  object
Gender                 int64
Age                    int64
GPA                  float64
Major                 object
Interested Domain     object
Projects              object
Future Career         object
Python                 int64
SQL                    int64
Java                   int64
Skill Score          float64
dtype: object

Missing values:
Student ID           0
Name                 0
Gender               0
Age                  0
GPA                  0
Major                0
Interested Domain    0
Projects             0
Future Career        0
Python               0
SQL                  0
Java                 0
Skill Score          0
dtype: int64



,Student ID,Name,Gender,Age,GPA,Major,Interested Domain,Projects,Future Career,Python,SQL,Java,Skill Score
0,1,John Smith,0,21,3.5,Computer Science,Artificial Intelligence,Chatbot Development,Machine Learning Researcher,3,3,1,2.33
1,2,Alice Johnson,1,20,3.2,Computer Science,Data Science,Data Analytics,Data Scientist,2,3,1,2.00
2,3,Robert Davis,0,22,3.8,Computer Science,Software Development,E-Commerce Website,Software Engineer,3,3,2,2.67
3,4,Emily Wilson,1,21,3.7,Computer Science,Web Development,Full-Stack Web App,Web Developer,1,3,3,2.33
4,5,Michael Brown,0,23,3.4,Computer Science,Cybersecurity,Network Security,Information Security Analyst,2,1,3,2.00


## Save the Cleaned Data

We save the cleaned dataframe to a new CSV file.

In [15]:
OUTPUT_FILE = 'cs_students_clean.csv'

df.to_csv(OUTPUT_FILE, index=False)

print(f'Saved {len(df)} rows to {OUTPUT_FILE}')

Saved 180 rows to cs_students_clean.csv
